In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import (
    StructType, StructField, IntegerType, LongType,
    DoubleType, StringType, TimestampNTZType
)

spark = SparkSession.builder.appName("NYC Taxi Data Raw Ingestion").getOrCreate()

taxi_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True),
])

taxi_df = spark.read.schema(taxi_schema).option("mode", "FAILFAST").parquet("/Volumes/nyc_tlc/landing/yellow_taxi_volume")

taxi_transformed_df = taxi_df.withColumns({
    "source_file" : f.col("_metadata.file_name"),
    "source_file_path" : f.col("_metadata.file_path"),
    "ingestion_timestamp" : f.current_timestamp()
})

taxi_transformed_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("nyc_bronze.bronze.yello_taxi_raw")